[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sharukh01621/first-line-to-first-paper/blob/main/colab_drive/solutions/part_07_research_capstone/ch36_flagship_research_capstone/02_guided_todos_SOLUTION.ipynb)

> **From First Line to First Paper** — *Mastering Python, Statistical Data Science, and Applied Deep Learning*  
> **Author:** Mezanur Rahman Sharukh | **Mode:** `ANNOTATED SOLUTION NOTEBOOK`


# Chapter 36 Guided To-Do Practice Labs

**Book Reference:** Chapter 36 (Grand Finale Capstone), Section 36.5 (Guided To-Do Practice)  
Run **Step 0** to activate `sharukh-tutor`, implement all 6 publication-pipeline building blocks below, and pass each **function name (without parentheses!)** to `tutor.check(...)`. Use `tutor.hint("ch36_todo_XX", level=1..3)` whenever you want a progressive nudge.

In [ ]:
# @title Step 0: Click Play (▶) to Activate Interactive Hints & Auto-Checker
!pip install -q "git+https://github.com/sharukh01621/first-line-to-first-paper.git#subdirectory=sharukh_tutor"
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import (
    HistGradientBoostingClassifier,
    RandomForestClassifier,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import brier_score_loss, roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn as nn
import sharukh_tutor as tutor

torch.set_num_threads(1)
mch_df = tutor.load_data("mch_survey_sample", as_pandas=True)
tutor.activate(chapter="ch36", notebook="guided_todos")


---
## To-Do 36.1 (Epidemiological Cohort Table): `build_publication_table1(mch_df)`

Clean `mch_df` without mutating it (`anc_visits == 999 -> NaN`, title-case `residence` -> `is_urban`, drop missing `child_stunted`), and compare `child_stunted == 0` vs `1` across:
- Continuous variables `["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]`: formatted `"mean ± SD"` (2 decimals) with Welch's t-test (`equal_var=False`) `p_value`.
- Binary variable `"is_urban"`: formatted `"count (pct%)"` (1 decimal) with Pearson chi-square (`correction=False`) `p_value`.
- Return DataFrame `["variable", "not_stunted", "stunted", "p_value"]` with `p_value` rounded to 4 decimals.

In [ ]:
def build_publication_table1(mch_df):
    """Build Journal Table 1 comparing non-stunted vs stunted."""
    df = mch_df.copy()
    anc = pd.to_numeric(df["anc_visits"], errors="coerce")
    df["anc_visits"] = anc.replace(999, np.nan)
    df["residence"] = (
        df["residence"].astype(str).str.strip().str.title()
    )
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    df = df.dropna(subset=["child_stunted"]).copy()
    df["child_stunted"] = df["child_stunted"].astype(int)

    g0 = df[df["child_stunted"] == 0]
    g1 = df[df["child_stunted"] == 1]
    rows = []
    for col in ["maternal_age_years", "anc_visits", "maternal_hb_g_dl"]:
        s0, s1 = g0[col].dropna(), g1[col].dropna()
        _, p_val = stats.ttest_ind(s0, s1, equal_var=False)
        rows.append({
            "variable": col,
            "not_stunted": f"{s0.mean():.2f} ± {s0.std(ddof=1):.2f}",
            "stunted": f"{s1.mean():.2f} ± {s1.std(ddof=1):.2f}",
            "p_value": round(float(p_val), 4),
        })

    ct = pd.crosstab(df["child_stunted"], df["is_urban"])
    _, p_chi, _, _ = stats.chi2_contingency(ct, correction=False)
    c0, pct0 = int(g0["is_urban"].sum()), 100.0 * g0["is_urban"].mean()
    c1, pct1 = int(g1["is_urban"].sum()), 100.0 * g1["is_urban"].mean()
    rows.append({
        "variable": "is_urban",
        "not_stunted": f"{c0} ({pct0:.1f}%)",
        "stunted": f"{c1} ({pct1:.1f}%)",
        "p_value": round(float(p_chi), 4),
    })
    return pd.DataFrame(rows)


In [ ]:
tutor.check("ch36_todo_01", build_publication_table1)


---
## To-Do 36.2 (4-Model Cluster Benchmark): `run_capstone_model_benchmark(mch_df, seed=42)`

Clean `mch_df` without mutating it, select the 5 core predictors `["maternal_age_years", "anc_visits", "maternal_hb_g_dl", "is_urban", "wealth_rank"]` (NEVER `child_haz`!), and evaluate 4 model families under strict 5-fold `GroupKFold` (`cluster_id`):
- `LogisticRegression`, `RandomForestClassifier`, `HistGradientBoostingClassifier`, and a compact PyTorch `MLP` (`Linear -> ReLU -> Linear`, 16 hidden units, `BCEWithLogitsLoss`, `Adam`, 30 epochs, train-only imputation & `StandardScaler` inside each fold!).
- Return DataFrame `["model", "mean_auc", "std_auc", "mean_brier"]` sorted by `mean_auc` descending, rounded to 4 decimals.

In [ ]:
def run_capstone_model_benchmark(mch_df, seed=42):
    """Benchmark LR, RF, HGB, and PyTorch MLP via 5-fold GroupKFold."""
    wealth_map = {
        "Poorest": 1, "Poorer": 2, "Middle": 3,
        "Richer": 4, "Richest": 5,
    }
    feats = [
        "maternal_age_years", "anc_visits",
        "maternal_hb_g_dl", "is_urban", "wealth_rank",
    ]
    df = mch_df.copy()
    anc = pd.to_numeric(df["anc_visits"], errors="coerce")
    df["anc_visits"] = anc.replace(999, np.nan)
    df["residence"] = (
        df["residence"].astype(str).str.strip().str.title()
    )
    df["is_urban"] = (df["residence"] == "Urban").astype(int)
    df["wealth_rank"] = df["wealth_quintile"].map(wealth_map)
    df = df.dropna(subset=["child_stunted"]).copy()
    X = df[feats].astype(float)
    y = df["child_stunted"].astype(int).to_numpy()
    groups = df["cluster_id"].to_numpy()
    gkf = GroupKFold(n_splits=5)

    builders = {
        "LogisticRegression": lambda: Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("scl", StandardScaler()),
            ("clf", LogisticRegression(max_iter=1000, random_state=seed)),
        ]),
        "RandomForestClassifier": lambda: Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", RandomForestClassifier(
                n_estimators=80, max_depth=4, random_state=seed
            )),
        ]),
        "HistGradientBoostingClassifier": lambda: Pipeline([
            ("imp", SimpleImputer(strategy="median")),
            ("clf", HistGradientBoostingClassifier(
                max_iter=50, max_depth=3, random_state=seed
            )),
        ]),
    }
    rows = []
    for name, make_pipe in builders.items():
        aucs, briers = [], []
        for tr, va in gkf.split(X, y, groups):
            pipe = make_pipe()
            pipe.fit(X.iloc[tr], y[tr])
            probs = pipe.predict_proba(X.iloc[va])[:, 1]
            aucs.append(roc_auc_score(y[va], probs))
            briers.append(brier_score_loss(y[va], probs))
        rows.append({
            "model": name,
            "mean_auc": round(float(np.mean(aucs)), 4),
            "std_auc": round(float(np.std(aucs)), 4),
            "mean_brier": round(float(np.mean(briers)), 4),
        })

    torch.set_num_threads(1)
    mlp_aucs, mlp_briers = [], []
    for fold_i, (tr, va) in enumerate(gkf.split(X, y, groups)):
        imp = SimpleImputer(strategy="median")
        scl = StandardScaler()
        X_tr_s = scl.fit_transform(imp.fit_transform(X.iloc[tr]))
        X_va_s = scl.transform(imp.transform(X.iloc[va]))
        torch.manual_seed(seed + fold_i)
        net = nn.Sequential(
            nn.Linear(X_tr_s.shape[1], 16),
            nn.ReLU(),
            nn.Linear(16, 1),
        )
        opt = torch.optim.Adam(net.parameters(), lr=0.02)
        loss_fn = nn.BCEWithLogitsLoss()
        xt = torch.tensor(X_tr_s, dtype=torch.float32)
        yt = torch.tensor(y[tr], dtype=torch.float32).unsqueeze(1)
        xv = torch.tensor(X_va_s, dtype=torch.float32)
        net.train()
        for _ in range(30):
            opt.zero_grad()
            loss_fn(net(xt), yt).backward()
            opt.step()
        net.eval()
        with torch.no_grad():
            probs = torch.sigmoid(net(xv)).squeeze(-1).numpy()
        mlp_aucs.append(roc_auc_score(y[va], probs))
        mlp_briers.append(brier_score_loss(y[va], probs))
    rows.append({
        "model": "PyTorch_MLP",
        "mean_auc": round(float(np.mean(mlp_aucs)), 4),
        "std_auc": round(float(np.std(mlp_aucs)), 4),
        "mean_brier": round(float(np.mean(mlp_briers)), 4),
    })
    return (
        pd.DataFrame(rows)
        .sort_values("mean_auc", ascending=False)
        .reset_index(drop=True)
    )


In [ ]:
tutor.check("ch36_todo_02", run_capstone_model_benchmark)


---
## To-Do 36.3 (Publication Results Markdown): `build_publication_results_markdown(leaderboard_df, fairness_dict)`

Return a clean Markdown string containing:
- `### Model Benchmark` followed by a Markdown pipe table with one row per model in `leaderboard_df`.
- `### Subgroup Fairness Summary` reporting `demographic_parity_diff` and `equal_opportunity_diff`.

In [ ]:
def build_publication_results_markdown(leaderboard_df, fairness_dict):
    """Generate clean Markdown benchmark table & fairness summary."""
    lines = [
        "### Model Benchmark",
        "",
        "| Model | Mean AUC | SD AUC | Mean Brier |",
        "| :--- | :---: | :---: | :---: |",
    ]
    for r in leaderboard_df.itertuples(index=False):
        lines.append(
            f"| {r.model} | {r.mean_auc:.4f} | "
            f"{r.std_auc:.4f} | {r.mean_brier:.4f} |"
        )
    dp = float(fairness_dict["demographic_parity_diff"])
    eo = float(fairness_dict["equal_opportunity_diff"])
    lines.extend([
        "",
        "### Subgroup Fairness Summary",
        "",
        f"- **Demographic Parity Diff:** {dp:.4f}",
        f"- **Equal Opportunity Diff:** {eo:.4f}",
    ])
    return "\n".join(lines)


In [ ]:
tutor.check("ch36_todo_03", build_publication_results_markdown)


---
## To-Do 36.4 (Decision Curve Analysis): `compute_clinical_net_benefit(y_true, y_prob, thresholds)`

Implement Vickers & Elkin (2006) clinical net benefit:
- Validate every `t` in `thresholds` satisfies `0 < t < 1` (raising `ValueError` otherwise).
- Compute `nb_model = (TP / N) - (FP / N) * (t / (1 - t))`, `nb_all = prev - (1 - prev) * (t / (1 - t))`, and `nb_none = 0.0`.
- Return DataFrame `["threshold", "nb_model", "nb_all", "nb_none"]` rounded to 5 decimals.

In [ ]:
def compute_clinical_net_benefit(y_true, y_prob, thresholds):
    """Compute Vickers & Elkin (2006) clinical net benefit table."""
    yt = np.asarray(y_true, dtype=int)
    yp = np.asarray(y_prob, dtype=float)
    n = len(yt)
    prev = float(np.mean(yt))
    rows = []
    for t in thresholds:
        t_f = float(t)
        if not (0.0 < t_f < 1.0):
            raise ValueError(
                f"Threshold {t_f} must satisfy 0 < t < 1"
            )
        pred = (yp >= t_f).astype(int)
        tp = int(np.sum((pred == 1) & (yt == 1)))
        fp = int(np.sum((pred == 1) & (yt == 0)))
        odds = t_f / (1.0 - t_f)
        nb_m = round((tp / n) - (fp / n) * odds, 5) + 0.0
        nb_a = round(prev - (1.0 - prev) * odds, 5) + 0.0
        rows.append({
            "threshold": round(t_f, 5),
            "nb_model": nb_m,
            "nb_all": nb_a,
            "nb_none": 0.0,
        })
    return pd.DataFrame(rows)


In [ ]:
tutor.check("ch36_todo_04", compute_clinical_net_benefit)


---
## To-Do 36.5 (Structured Abstract): `draft_structured_abstract(study_stats)`

Format a 4-section structured medical research abstract (`**Background:**`, `**Methods:**`, `**Results:**`, `**Conclusions:**`) using all 6 required keys in `study_stats` (`n_total`, `stunting_pct`, `best_model`, `best_auc`, `brier_score`, `eo_gap`), count words (`len(abstract_text.split())`), and return `{"abstract_text": str, "word_count": int, "within_limit": bool}` (`word_count <= 300`).

In [ ]:
def draft_structured_abstract(study_stats):
    """Draft a concise 4-section medical abstract (<=300 words)."""
    n_tot = study_stats["n_total"]
    stunt_pct = float(study_stats["stunting_pct"])
    best_m = study_stats["best_model"]
    best_auc = float(study_stats["best_auc"])
    brier = float(study_stats["brier_score"])
    eo_gap = float(study_stats["eo_gap"])
    text = (
        f"**Background:** Childhood stunting remains a pressing "
        f"public health challenge requiring accurate risk "
        f"stratification.\n\n"
        f"**Methods:** We evaluated {n_tot} survey households "
        f"(stunting prevalence {stunt_pct:.1f}%) under 5-fold "
        f"cluster-grouped cross-validation with Decision Curve "
        f"Analysis and rural-urban fairness auditing.\n\n"
        f"**Results:** {best_m} achieved top discrimination "
        f"(ROC-AUC {best_auc:.4f}; Brier {brier:.4f}), while "
        f"subgroup auditing identified an equal-opportunity gap "
        f"of {eo_gap:.4f}.\n\n"
        f"**Conclusions:** Leak-free cluster validation paired with "
        f"fairness reporting supports responsible screening."
    )
    wc = len(text.split())
    return {
        "abstract_text": text,
        "word_count": wc,
        "within_limit": bool(wc <= 300),
    }


In [ ]:
tutor.check("ch36_todo_05", draft_structured_abstract)


---
## To-Do 36.6 (Reproducibility Gate): `audit_manuscript_reproducibility(release_dir)`

Inspect `Path(release_dir)` for all 6 required release artifacts (`MODEL_CARD.md`, `fairness_audit.json`, `leaderboard.csv`, `manifest.json`, `model.joblib`, `table1.csv`) and return `{"present": sorted_list, "missing": sorted_list, "ready_for_submission": bool}`.

In [ ]:
def audit_manuscript_reproducibility(release_dir):
    """Audit release_dir for all 6 required submission artifacts."""
    required = sorted([
        "MODEL_CARD.md", "fairness_audit.json", "leaderboard.csv",
        "manifest.json", "model.joblib", "table1.csv",
    ])
    root = Path(release_dir)
    present = sorted(f for f in required if (root / f).is_file())
    missing = sorted(f for f in required if not (root / f).is_file())
    return {
        "present": present,
        "missing": missing,
        "ready_for_submission": len(missing) == 0,
    }


In [ ]:
tutor.check("ch36_todo_06", audit_manuscript_reproducibility)


---
## Annotated Debugging Post-Mortem (Solution Notebook Only)

- **To-Do 36.1 (Sentinel `999` & Lowercase `"urban"` Traps):** Leaving `anc_visits == 999` inflates mean ANC visits from `3.90` to `19.70`, while skipping `.str.title()` misses 4 lowercase `"urban"` households and shifts the chi-square p-value.
- **To-Do 36.2 (Preventing `child_haz` Leakage & Fold Leakage):** Including `child_haz` leaks the target (`child_stunted = child_haz < -2`) and artificially drives AUC near `1.00`. Likewise, fitting `SimpleImputer` or `StandardScaler` outside `GroupKFold` leaks cluster statistics across splits.
- **To-Do 36.4 (DCA Threshold Boundary Guard):** Dividing `t / (1 - t)` at `t = 1.0` triggers `ZeroDivisionError` (or infinite penalty); always guard `0 < t < 1` with `ValueError`.

In [ ]:
tutor.summary()
